# Tail dependence — does a signal work only via rare hauls, or on the steady body too?
_Read-only diagnostic: each signal's rank-association with points with and without haul gameweeks, per position, with the tail-sensitivity flag and haul-frequency context. **Association only.** DGW excluded._

**Sections:** (a) full vs ex-haul rho (tail-sensitivity) · (b) haul frequency context

---

## Setup
> Whole season, `minutes > 0`, **DGW excluded**; per position, leading-indicator universe (exact composites dropped). Full vs ex-haul rho via `measure_tail_event_dependence` (a haul is `total_points > 12`). A signal whose association collapses once hauls are removed is a boom-bust signal; one that holds is a steady-floor signal.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from dal.pipeline import load as load_mart, run as run_pipeline
from dal.exceptions import MartNotBuiltError, MartSchemaError
from research.kernels.descriptive.relevance import (
    compute_relevance, leading_indicator_signals, leading_alive_signals, POSITIONS,
)
from research.kernels.diagnostic.tail import measure_tail_event_dependence

POSITIONS = list(POSITIONS)
COLOURS = {"GK": "#9467bd", "DEF": "#1f77b4", "MID": "#2ca02c", "FWD": "#d62728"}

try:
    _r = load_mart()
except (MartNotBuiltError, MartSchemaError) as _e:
    print(f"Rebuilding mart ({type(_e).__name__})...")
    run_pipeline(force=True)
    _r = load_mart()

mart = _r.mart
df = mart[mart["gw"].between(1, _r.data_cutoff_gw)].copy()
df = df[df["minutes"].notna() & (df["minutes"] > 0)].copy()
df = df[df["is_dgw"] == False].copy()

leading = sorted(leading_indicator_signals(drop_exact_composites=True))
alive_by_pos = {}
for p in POSITIONS:
    rel = compute_relevance(df[df["position"] == p], signals=leading, group_cols=())
    alive_by_pos[p] = leading_alive_signals(rel)

print(f"Study range: GW 1 - {_r.data_cutoff_gw} | minutes > 0 | DGW excluded | n = {len(df):,}")
for p in POSITIONS:
    print(f"  {p}: n={len(df[df.position == p]):>6,} | {len(alive_by_pos[p])} live signals")

## (a) Full vs ex-haul rho
> The grey dot is the full association; the coloured dot recomputes it with haul gameweeks removed. A large drop (flagged `tail_sensitive`, shown red) means the signal mostly 'works' through rare explosions — a captaincy gamble rather than a steady-points read. `tail_sensitive = None` (too few hauls to assess) is never read as safe.

In [ ]:
tl_rows = []
for p in POSITIONS:
    for sig in alive_by_pos[p]:
        tl_rows.append({"position": p, "signal": sig,
                        **measure_tail_event_dependence(df, sig, "total_points", p)})
tail = pd.DataFrame(tl_rows)
display(tail.sort_values(["position", "rho_full"], ascending=[True, False]).round(4))

fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharex=True)
for ax, p in zip(axes, POSITIONS):
    sub = tail[(tail.position == p) & tail["support_flag"].eq("") & tail["rho_no_haul"].notna()].sort_values("rho_full")
    yv = np.arange(len(sub))
    ts = sub["tail_sensitive"].fillna(False).astype(bool).to_numpy()
    ax.hlines(yv, sub["rho_no_haul"], sub["rho_full"], color="lightgrey", zorder=1)
    ax.scatter(sub["rho_full"], yv, color="#bdbdbd", label="full", zorder=2)
    ax.scatter(sub["rho_no_haul"], yv, color=np.where(ts, "#d62728", COLOURS[p]), label="ex-haul", zorder=3)
    ax.set_yticks(yv)
    ax.set_yticklabels(sub["signal"], fontsize=7)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_title(p)
    ax.set_xlabel("Spearman rho vs total_points")
axes[0].legend(fontsize=8, loc="lower right")
fig.suptitle("Full -> ex-haul: red = tail-sensitive (association leans on rare hauls)", y=1.03)
plt.tight_layout()
plt.show()

## (b) Haul frequency context
> Every ex-haul drop must be read against how often hauls actually happen. `haul_pct` is the share of featured gameweeks with `total_points > 12` (a property of position, not of the signal), with the raw count `n_haul`.

In [ ]:
ctx = tail.groupby("position")[["haul_pct", "n_haul"]].first().reindex(POSITIONS)
display(ctx)

fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(ctx.index, ctx["haul_pct"], color=[COLOURS[p] for p in ctx.index])
for i, p in enumerate(ctx.index):
    ax.text(i, ctx["haul_pct"][p], f"n={int(ctx['n_haul'][p])}", ha="center", va="bottom", fontsize=8)
ax.set_ylabel("haul % of featured GWs")
ax.set_title("Haul frequency by position (total_points > 12)")
plt.tight_layout()
plt.show()